In [3]:
import pandas as pd
import numpy as np
import os
import joblib

from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
# ==========================================
# 1. LOAD PROCESSED DATA
# ==========================================

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("===== DATA LOADED =====")
print("Training features:", X_train.shape)
print("Testing features :", X_test.shape)
print("Training labels  :", y_train.shape)
print("Testing labels   :", y_test.shape)


# ==========================================
# 2. PREPARE TARGET FOR XGBOOST
# ==========================================

# Dataset labels are:
# 1 = Normal
# 2 = Suspect
# 3 = Pathological
#
# XGBoost uses class labels starting at 0,
# so convert 1,2,3 -> 0,1,2

y_train_xgb = y_train.astype(int) - 1
y_test_xgb = y_test.astype(int) - 1


# ==========================================
# 3. CREATE XGBOOST MODEL
# ==========================================

print("\n===== CREATING XGBOOST =====")

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)


# ==========================================
# 4. TRAIN
# ==========================================

print("\n===== TRAINING XGBOOST =====")

model.fit(
    X_train,
    y_train_xgb
)

print("Training complete!")


# ==========================================
# 5. PREDICTIONS
# ==========================================

y_pred_xgb = model.predict(X_test)

# Convert 0,1,2 back to 1,2,3
y_pred = y_pred_xgb + 1


# ==========================================
# 6. EVALUATION
# ==========================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("\n===== XGBOOST RESULTS =====")

print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Macro F1 : {macro_f1:.4f} ({macro_f1*100:.2f}%)")

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Normal",
            "Suspect",
            "Pathological"
        ]
    )
)

print("\nConfusion Matrix:")

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)


# ==========================================
# 7. FEATURE IMPORTANCE
# ==========================================

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\n===== TOP 15 FEATURES =====")

print(
    importance.head(15).to_string(index=False)
)


# ==========================================
# 8. SAVE MODEL
# ==========================================

os.makedirs("../models", exist_ok=True)

joblib.dump(
    model,
    "../models/xgboost.pkl"
)


# ==========================================
# 9. SAVE FEATURE IMPORTANCE
# ==========================================

os.makedirs("../results", exist_ok=True)

importance.to_csv(
    "../results/xgboost_feature_importance.csv",
    index=False
)


# ==========================================
# 10. FINAL MESSAGE
# ==========================================

print("\n===== SAVED =====")
print("Model:")
print("../models/xgboost.pkl")

print("\nFeature importance:")
print("../results/xgboost_feature_importance.csv")

===== DATA LOADED =====
Training features: (1700, 30)
Testing features : (426, 30)
Training labels  : (1700,)
Testing labels   : (426,)

===== CREATING XGBOOST =====

===== TRAINING XGBOOST =====
Training complete!

===== XGBOOST RESULTS =====
Accuracy : 0.9413 (94.13%)
Macro F1 : 0.8922 (89.22%)

Classification Report:
              precision    recall  f1-score   support

      Normal       0.96      0.98      0.97       332
     Suspect       0.87      0.76      0.81        59
Pathological       0.91      0.89      0.90        35

    accuracy                           0.94       426
   macro avg       0.91      0.88      0.89       426
weighted avg       0.94      0.94      0.94       426


Confusion Matrix:
[[325   6   1]
 [ 12  45   2]
 [  3   1  31]]

===== TOP 15 FEATURES =====
Feature  Importance
   MSTV    0.123150
   Mean    0.099563
   DP.1    0.098100
   ASTV    0.085490
     AC    0.079768
   ALTV    0.076558
     DP    0.067041
   AC.1    0.056590
 Median    0.030560
   